Import all the required libraries

NLTK => Natural Language Toolkit

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

In [2]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

Download nltk resources

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

Data Creation

In [51]:
data = {
    "review": [
        "I have not look teacher like you.",
        "I feel boring class.",
        "Awseome mentor...very intelligent.",
        "I love your teaching style.",
        "This topic was too boring.",
        "Very disappointed.",
        "Waste of money.",
        "Your are elaborating short topic very long.",
        "Awesome, I learn more.",
        "Excellent value for money.",
        "Mentor is not responding in live chat."
    ],


    "sentiment": [
        1,
        0,
        1,
        1,
        0,
        0,
        0,
        0,
        1,
        1,
        0
    ]
}

In [52]:
df = pd.DataFrame(data)

In [53]:
df.head()

,review,sentiment
0,I have not look teacher like you.,1
1,I feel boring class.,0
2,Awseome mentor...very intelligent.,1
3,I love your teaching style.,1
4,This topic was too boring.,0


In [54]:
df['sentiment'].value_counts()

,count
sentiment,
0,6
1,5


Text Preprocessing

```python
# Create the mapping table
table = str.maketrans("aeiou", "12345")

# Translate the string
text = "hello world"
print(text.translate(table))  # Output: h2ll4 w4rld
```

In [55]:
import string
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def preprocess(text):
  # 1. lowercase
  text = text.lower()

  # 2. remove punctuation
  text = text.translate(
      str.maketrans('', '', string.punctuation)
  )

  # 3. remove numbers
  text = re.sub(r'\d+', '', text)

 # 4. tokenization
  words = text.split()

  # stopword removal + stemming
  words = [
      # stemming => find the root word for the all the above tokens
      stemmer.stem(word)
      for word in words if word not in stop_words
  ]

  return " ".join(words)

In [56]:
# function calling
df['clean_review'] = df["review"].apply(preprocess)
df

,review,sentiment,clean_review
0,I have not look teacher like you.,1,look teacher like
1,I feel boring class.,0,feel bore class
2,Awseome mentor...very intelligent.,1,awseom mentorveri intellig
3,I love your teaching style.,1,love teach style
4,This topic was too boring.,0,topic bore
5,Very disappointed.,0,disappoint
6,Waste of money.,0,wast money
7,Your are elaborating short topic very long.,0,elabor short topic long
8,"Awesome, I learn more.",1,awesom learn
9,Excellent value for money.,1,excel valu money


In [57]:
import string

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text):
  # 1. lowercase
  text = text.lower()

  # 2. remove punctuation
  text = text.translate(
      str.maketrans('', '', string.punctuation)
  )

  # 3. remove numbers
  text = re.sub(r'\d+', '', text)

 # 4. tokenization
  words = text.split()

  # stopword removal
  words = [word for word in words if word not in stop_words]

  # lemmatization
  words = [lemmatizer.lemmatize(word) for word in words]

  return " ".join(words)

In [58]:
# function calling
df['clean_review_lemma'] = df["review"].apply(preprocess)
df

,review,sentiment,clean_review,clean_review_lemma
0,I have not look teacher like you.,1,look teacher like,look teacher like
1,I feel boring class.,0,feel bore class,feel boring class
2,Awseome mentor...very intelligent.,1,awseom mentorveri intellig,awseome mentorvery intelligent
3,I love your teaching style.,1,love teach style,love teaching style
4,This topic was too boring.,0,topic bore,topic boring
5,Very disappointed.,0,disappoint,disappointed
6,Waste of money.,0,wast money,waste money
7,Your are elaborating short topic very long.,0,elabor short topic long,elaborating short topic long
8,"Awesome, I learn more.",1,awesom learn,awesome learn
9,Excellent value for money.,1,excel valu money,excellent value money


Convert the text into numeric vector via Bag Of Words using CountVectorizer

In [59]:
cv = CountVectorizer()
X = cv.fit_transform(df["clean_review_lemma"])
y = df['sentiment']

In [60]:
cv.vocabulary_

{'look': 14,
 'teacher': 22,
 'like': 11,
 'feel': 8,
 'boring': 2,
 'class': 4,
 'awseome': 1,
 'mentorvery': 17,
 'intelligent': 9,
 'love': 15,
 'teaching': 23,
 'style': 21,
 'topic': 24,
 'disappointed': 5,
 'waste': 26,
 'money': 18,
 'elaborating': 6,
 'short': 20,
 'long': 13,
 'awesome': 0,
 'learn': 10,
 'excellent': 7,
 'value': 25,
 'mentor': 16,
 'responding': 19,
 'live': 12,
 'chat': 3}

In [61]:
cv.get_feature_names_out()

array(['awesome', 'awseome', 'boring', 'chat', 'class', 'disappointed',
       'elaborating', 'excellent', 'feel', 'intelligent', 'learn', 'like',
       'live', 'long', 'look', 'love', 'mentor', 'mentorvery', 'money',
       'responding', 'short', 'style', 'teacher', 'teaching', 'topic',
       'value', 'waste'], dtype=object)

Convert the matrix into dataframe

In [62]:
bow_df = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out())
bow_df.head()

,awesome,awseome,boring,chat,class,disappointed,elaborating,excellent,feel,intelligent,...,mentorvery,money,responding,short,style,teacher,teaching,topic,value,waste
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,0,0,1,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [63]:
bow_df

,awesome,awseome,boring,chat,class,disappointed,elaborating,excellent,feel,intelligent,...,mentorvery,money,responding,short,style,teacher,teaching,topic,value,waste
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,0,0,1,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
7,0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,1,0,0
8,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,1,0


Train-Test Split

In [64]:
X_train, X_test, y_train, y_test = train_test_split(bow_df, y, test_size=0.2, random_state=44)

Model Training

In [65]:
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

Model Prediction

In [66]:
predictions = model.predict(X_test)
print(predictions)

[0 0 0]


In [67]:
X_test

,awesome,awseome,boring,chat,class,disappointed,elaborating,excellent,feel,intelligent,...,mentorvery,money,responding,short,style,teacher,teaching,topic,value,waste
8,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,1,0,0
2,0,1,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0


In [68]:
accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.3333333333333333


In [69]:
cm = confusion_matrix(y_test, predictions)
print(cm)

[[1 0]
 [2 0]]


In [70]:
review = "Your teaching style is excellent."
clean = preprocess(review)
vector = cv.transform([clean])
prediction = model.predict(vector)
print(prediction)

[1]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MultinomialNB was fitted with feature names
  warnings.warn(


In [71]:
if prediction[0] == 1:
  print("Positive Review")
else:
  print("Negative Review")

Positive Review
